In [1]:
import ast
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from train_utils import get_metrics
from scipy.optimize import minimize

In [2]:
models = [
'c1-442-d3bm21p4-512-12-22-6e51e3-C099020d0-rmse',
'c1-442-d3lm21p4-512-8-22-5e51e3-C099050d0-rmse',
'c1-442-dbm21p4-512-12-22-6e51e3-C099020d0-rmse',
'c1-442-dlm21p4-512-6-22-5e51e3-C099050d0-rmse',
'c1-442-rbm21p4-512-12-22-6e51e3-C099020d0-rmse',
'c1-442-rlm21p4-512-12-22-5e51e3-C099050d0-rmse',
        ] 

n_folds=4

In [3]:
data_set_df_list=[]
for i in models:
    folds_list=[]
    for fold in range(n_folds):
        folds_list.append(pd.read_csv(f"../../output/commonlit-evaluate-student-summaries/{i}/predictions_fold{fold}.csv"))
    fold_df = pd.concat(folds_list, axis=0)
    data_set_df_list.append(fold_df)
    
print(len(data_set_df_list), data_set_df_list[0].shape)
data_set_df_list[0].head()

6 (7165, 17)


,student_id,prompt_id,text,content,wording,prompt_question,prompt_title,prompt_text,fold,prompt_title_processed,prompt_question_processed,prompt_text_processed,text_processed,full_text_processed,full_text_len,content_predictions,wording_predictions
0,ad7245db300c,39c16e,The main person is a likable person that is neither good nor bad but his fortune goes from good to bad from an error or frailty,-1.547163,-1.461245,"Summarize at least 3 elements of an ideal tragedy, as described by Aristotle.",On Tragedy,"Chapter 13 \r\nAs the sequel to what has already been said, we must proceed to consider what the poet should aim at, and what he should avoid, in constructing his plots; and by what means the specific effect of Tragedy will be produced. \r\nA perfect tragedy should, as we have seen, be arranged not on the simple but on the complex plan. It should, moreover, imitate actions which excite pity and fear, this being the distinctive mark of tragic imitation. It follows plainly, in the first place, that the change of fortune presented must not be the spectacle of a virtuous man brought from prosp...",0,On Tragedy,"Summarize at least 3 elements of an ideal tragedy, as described by Aristotle.","Chapter 13 [BR] As the sequel to what has already been said, we must proceed to consider what the poet should aim at, and what he should avoid, in constructing his plots; and by what means the specific effect of Tragedy will be produced. [BR] A perfect tragedy should, as we have seen, be arranged not on the simple but on the complex plan. It should, moreover, imitate actions which excite pity and fear, this being the distinctive mark of tragic imitation. It follows plainly, in the first place, that the change of fortune presented must not be the spectacle of a virtuous man brought from pro...",The main person is a likable person that is neither good nor bad but his fortune goes from good to bad from an error or frailty,"Content Wording [SEP] On Tragedy [Question] Summarize at least 3 elements of an ideal tragedy, as described by Aristotle. [Answer] The main person is a likable person that is neither good nor bad but his fortune goes from good to bad from an error or frailty",55,-1.450244,-1.523960
1,ac8891e90289,39c16e,"a complex twisting plot that makes you feel pity for the character and fear of becoming the pitied, also the change in fortune of the charter",-1.547163,-1.461245,"Summarize at least 3 elements of an ideal tragedy, as described by Aristotle.",On Tragedy,"Chapter 13 \r\nAs the sequel to what has already been said, we must proceed to consider what the poet should aim at, and what he should avoid, in constructing his plots; and by what means the specific effect of Tragedy will be produced. \r\nA perfect tragedy should, as we have seen, be arranged not on the simple but on the complex plan. It should, moreover, imitate actions which excite pity and fear, this being the distinctive mark of tragic imitation. It follows plainly, in the first place, that the change of fortune presented must not be the spectacle of a virtuous man brought from prosp...",0,On Tragedy,"Summarize at least 3 elements of an ideal tragedy, as described by Aristotle.","Chapter 13 [BR] As the sequel to what has already been said, we must proceed to consider what the poet should aim at, and what he should avoid, in constructing his plots; and by what means the specific effect of Tragedy will be produced. [BR] A perfect tragedy should, as we have seen, be arranged not on the simple but on the complex plan. It should, moreover, imitate actions which excite pity and fear, this being the distinctive mark of tragic imitation. It follows plainly, in the first place, that the change of fortune presented must not be the spectacle of a virtuous man brought from pro...","a complex twisting plot that makes you feel pity for the character and fear of becoming the pitied, also the change in fortune of the charter","Content Wording [SEP] On Tragedy [Question] Summarize at least

In [4]:
def get_equation(x, p):
    comps = []
    for idx, i in enumerate(p):
        i = np.vstack(i)
        out = i * x[idx]
        comps.append(out)
    eqn = sum(comps)
    return eqn

def f(x, p):
    pred1 = get_equation(x, p)
#     print(pred1.shape)
    classwise_metric, overall_metric = get_metrics(torch.tensor(pred1), torch.tensor(final_df[label_cols].values))
    return overall_metric

def optimize_score(n):
    p = preds[:n]
    weight_init = [1 for _ in range(len(p))]
    
    result = minimize(f, weight_init, args=p, method="Nelder-Mead")
    return result['x'], result['fun']

In [5]:
label_cols = ["content", "wording"]

final_df = data_set_df_list[0]
preds = []
for idx, df in enumerate(data_set_df_list):
    for label in label_cols:
        final_df[f'{label}_predictions_{idx}'] = final_df['student_id'].map(df.set_index('student_id')[f'{label}_predictions'])
    pred_cols = [f"{col}_predictions_{idx}" for col in label_cols]    
    preds.append(final_df[pred_cols].values)
final_df.head()

pd.set_option('display.max_colwidth', None)
n=len(data_set_df_list)
# print(n)
scores = []
for i in range(n):
#     print(i)
    weight, score  = optimize_score(i+1)
    scores.append((score, weight))
display(pd.DataFrame(scores))

,0,1
0,0.537947,[1.01484375]
1,0.519774,"[0.4156817045344159, 0.6319900305212296]"
2,0.517682,"[0.48448299301024383, 0.7637415141040962, -0.2010484484370789]"
3,0.517030,"[0.4736402095435008, 0.6565757058607872, -0.2415362737222119, 0.16108816116309504]"
4,0.516041,"[0.4651089427984977, 0.7008473171929651, -0.21157183017002118, 0.2520641498519185, -0.14657945056310245]"
5,0.515483,"[0.44156833981958465, 0.7365164665441468, -0.15709296775105847, 0.2965906573183049, -0.13191138230524024, -0.12457739566343137]"


In [6]:
label_cols = ["content", "wording"]

final_df = data_set_df_list[0]
preds = []
for idx, df in enumerate(data_set_df_list):
    for label in label_cols:
        final_df[f'{label}_predictions_{idx}'] = final_df['student_id'].map(df.set_index('student_id')[f'{label}_predictions'])
    pred_cols = [f"{col}_predictions_{idx}" for col in label_cols]    
    preds.append(final_df[pred_cols].values)
final_df.head()

pd.set_option('display.max_colwidth', None)
n=len(data_set_df_list)
# print(n)
scores = []
for i in range(n):
#     print(i)
    weight, score  = optimize_score(i+1)
    scores.append((score, weight))
display(pd.DataFrame(scores))

,0,1
0,0.537947,[1.01484375]
1,0.519774,"[0.4156817045344159, 0.6319900305212296]"
2,0.517682,"[0.48448299301024383, 0.7637415141040962, -0.2010484484370789]"
3,0.517030,"[0.4736402095435008, 0.6565757058607872, -0.2415362737222119, 0.16108816116309504]"
4,0.516041,"[0.4651089427984977, 0.7008473171929651, -0.21157183017002118, 0.2520641498519185, -0.14657945056310245]"
5,0.515483,"[0.44156833981958465, 0.7365164665441468, -0.15709296775105847, 0.2965906573183049, -0.13191138230524024, -0.12457739566343137]"
